# Gain Power Calibration

This code maps the qick gain parameter to actual output power with oscilloscope

## Instrument Instantiation

In [7]:
import numpy as np
import math
import matplotlib.pyplot as plt
import time
import pyvisa
from typing import Union, Any
from pprint import pprint

from qcodes import (
    Parameter,
    Measurement,
    Station,
    load_or_create_experiment,
    initialise_or_create_database_at,
)
from qick import *
from qick.averager_program import QickSweep
from qick.pyro import make_proxy

# Qick version : 0.2.357
(soc, soccfg) = make_proxy("192.168.2.99")

# Set DAC Channel 0 attenuation 20 dB and 20 dB, and turn on DAC channel
soc.rfb_set_gen_rf(0,0,0)
# Set DAC Channel filter as bypass mode
soc.rfb_set_gen_filter(0,fc = 2.5, ftype = "lowpass")

# Set ADC Channel attenuation 20 dB, and turn on ADC channel
soc.rfb_set_ro_rf(0,0)
# Set ADC Channel filter as bypass mode
soc.rfb_set_ro_filter(0, fc = 2.5, ftype = "lowpass")

# Connect to oscilloscope
rm = pyvisa.ResourceManager()
visa_addr = "USB0::0x0957::0x1780::MY60101437::0::INSTR"
oscilloscope = rm.open_resource(visa_addr)
print(oscilloscope.query("*IDN?"))
oscilloscope.write(":TIMebase:RANGe 0.00005")

Pyro.NameServer PYRO:Pyro.NameServer@0.0.0.0:8888
myqick PYRO:obj_8fd72669100a4219ac4728a5316556f8@192.168.2.99:41081
AGILENT TECHNOLOGIES,DSO-X 6004A,MY60101437,07.31.2020012834



25

## QCodes Initialization

In [8]:
station = Station()

initialise_or_create_database_at("C:/Users/Measurement6/Nextcloud2/Lab/Data/QSim/2025/20251127_QICK_Test/gain_pwr_calb.db")
exp = load_or_create_experiment("2D sweep", "RF_Out_200MHz")
meas = Measurement(exp=exp, station=station)

meas_gain = Parameter(name = "gain", label = "gain", unit = "AU")
meas_freq = Parameter(name = "freq", label = "freq", unit = "Hz")
meas_pwr = Parameter(name = "pwr", label = "pwr", unit = "dBm")
meas.register_parameter(meas_gain)
meas.register_parameter(meas_freq)
meas.register_parameter(meas_pwr, setpoints=(meas_gain, meas_freq))

## PowerMeasurement Code

In [9]:
class PowerMeas(AveragerProgram):
    def initialize(self):
        freq_rf     = self.cfg["freq_rf"]
        # Declare RF generation channel
        self.declare_gen(
            ch      = 0,        # Channel
            nqz     = 1         # Nyquist Zone
        )
        # Declare RF input channel
        self.declare_readout(
            ch      = 0,        # Channel
            length  = self.cfg["duration"] + 100       # Readout length
        )
        # Convert RF frequency to DAC DDS register value
        freq_dac    = self.freq2reg(
            f       = freq_rf,  # Frequency
            gen_ch  = 0,        # Generator channel
            ro_ch   = 0         # Readout channel for round up
        )
        # Convert RF frequency to ADC DDS register value
        freq_adc    = self.freq2reg_adc(
            f       = freq_rf,  # Frequency
            ro_ch   = 0,        # Readout channel
            gen_ch  = 0         # Generator channel for round up
        )

        # Set DAC DDS
        self.set_pulse_registers(
            ch      = 0,        # Generator channel
            style   = "const",  # Output is gain * DDS output
            freq    = freq_dac, # Generator DDS frequency
            phase   = 0,        # Generator DDS phase
            gain    = self.cfg["gain"],      # Generator amplitude
            length  = self.cfg["duration"], # Pulse length
            phrst   = 0,        # Generator DDS phase reset
            mode    = "periodic"
        )
        # Set ADC DDS
        self.set_readout_registers(
            ch      = 0,        # Readout channel
            freq    = freq_adc, # Readout DDS frequency
            length  = self.cfg["duration"], # Readout DDS multiplication length
            phrst   = 0         # Readout DDS phase reset
        )
        self.synci(100)

    def body(self):
        self.pulse(
            ch      = 0,        # Generator channel
            t       = 100       # Pulse will be output @ sync_t + 100
        )
        self.readout(
            ch      = 0,        # Readout channel
            t       = 100       # Readout DDS will start multiplication
                                # @ sync_t + 100
        )
        self.trigger(
            adcs    = [0],      # Readout channels
            adc_trig_offset = 50 # Readout will capture the data @ sync_t + 50
        )
        self.sync_all(100)

def measure_pwr(
    freq:Union[float,int],
    gain:int,
    soc:Any,
    soccfg:QickConfig
) -> float:
    oscilloscope.write_termination = "\n"
    oscilloscope.read_termination = "\n"

    # Set input impedance
    oscilloscope.write(":RUN")
    oscilloscope.write(":CHAN2:IMP FIFT")
    freq = int(freq)
    cfg = {
        # Experiment Setup
        "reps" : 1,
        "duration" : 1500,
        "expts" : 1,
        "freq_rf" : freq,
        "gain" : gain
    }
    prog = PowerMeas(
        soccfg,
        cfg
    )
    prog.acquire(soc, progress = False)

    # Set function operation as fft
    oscilloscope.write(":FUNC1:DISP ON")
    oscilloscope.write(":FUNC1:OPER FFT")
    oscilloscope.write(":FUNC1:SOUR CHAN2")
    oscilloscope.write(f":FUNC1:CENT {freq} MHz")
    oscilloscope.write(":FUNC1:SPAN 100 MHz")
    oscilloscope.write(":SYST:PREC ON")
    oscilloscope.write(":MARK:MODE WAV")
    oscilloscope.write(":MARK:X1Y1Source MATH1")
    oscilloscope.write(f":MARK:X1Position {freq} MHz")
    time.sleep(3)
    pwr = 0
    for i in range(100):
        pwr += float(oscilloscope.query(":MARK:Y1Position?"))
    pwr = pwr / 100

    return pwr

## Power Measurement

In [28]:
import json
# Qick version : 0.2.357
(soc, soccfg) = make_proxy("192.168.2.99")


with meas.run() as datasaver:
    att1 = 0
    att2 = 0
    # Set DAC Channel 0 attenuation
    soc.rfb_set_gen_rf(0, att1, att2)
    # Set DAC Channel filter as bypass mode
    soc.rfb_set_gen_filter(0,fc = 2.5, ftype = "lowpass")
    soc.rfb_set_gen_filter(2,fc = 2.5, ftype = "lowpass")
    # Set ADC Channel attenuation 31 dB, and turn on ADC channel
    soc.rfb_set_ro_rf(0,0)
    # Set ADC Channel filter as bypass mode
    soc.rfb_set_ro_filter(0, fc = 2.5, ftype = "lowpass")
    datasaver.dataset.add_metadata(
        tag = "Attenuation",
        metadata = json.dumps(
            {
                "att1" : att1,
                "att2" : att2,
            }
        )
    )


    frequencies = np.linspace(180, 220, 3)
    gains = np.logspace(np.log10(10),np.log10(32767),30).astype(int)
    data = {}

    for freq in frequencies:
        print(f"Measureing {freq} Hz...")
        data = []
        for gain in gains:
            data.append(
                measure_pwr(
                    freq = freq,
                    gain = gain,
                    soc = soc,
                    soccfg = soccfg
                )
            )
            print(f"gain {gain} is measured...")
        datasaver.add_result(
            (meas_freq, [freq] * len(gains)),
            (meas_gain, gains),
            (meas_pwr, data)
        )


Pyro.NameServer PYRO:Pyro.NameServer@0.0.0.0:8888
myqick PYRO:obj_5160689e54084b35810407b9b3c9280c@192.168.2.99:41739
Starting experimental run with id: 6. 
Measureing 180.0 Hz...
gain 10 is measured...
gain 13 is measured...
gain 17 is measured...
gain 23 is measured...
gain 30 is measured...
gain 40 is measured...
gain 53 is measured...
gain 70 is measured...
gain 93 is measured...
gain 123 is measured...
gain 163 is measured...
gain 215 is measured...
gain 284 is measured...
gain 376 is measured...
gain 497 is measured...
gain 658 is measured...
gain 870 is measured...
gain 1150 is measured...
gain 1520 is measured...
gain 2010 is measured...
gain 2657 is measured...
gain 3512 is measured...
gain 4643 is measured...
gain 6139 is measured...
gain 8115 is measured...
gain 10728 is measured...
gain 14183 is measured...
gain 18749 is measured...
gain 24786 is measured...
gain 32766 is measured...
Measureing 200.0 Hz...
gain 10 is measured...
gain 13 is measured...
gain 17 is measured...

# Turn off RF

In [11]:
measure_pwr(
    freq = 200,
    gain = 0,
    soc = soc,
    soccfg = soccfg
)

-59.22343750000166